In [18]:
# Part 4:
# Setup
!pip install crewai crewai-tools -q
!pip install litellm -q
!pip install nest_asyncio -q

import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata
import os

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# Fix for a known CrewAI bug: disable an internal caching feature that Groq doesn't support
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

print("Setup complete")

Setup complete


In [19]:
# Creating the tools our agent can use
# A "tool" is just a normal Python function that the agent is allowed to call
from crewai.tools import tool
import requests

# Tool 1: this tool goes to a real website (API) on the internet and gets a joke
@tool("Get Random Joke")
def get_random_joke(topic: str = "any") -> str:
    """Fetches a random joke from a live joke API. Use this when the user wants a joke or something funny. The topic parameter is optional."""
    # send a request to the joke website
    response = requests.get("https://official-joke-api.appspot.com/random_joke")
    # turn the reply into something Python can read
    data = response.json()
    # build a simple sentence: setup + punchline
    joke = data['setup'] + " — " + data['punchline']
    return joke


# Tool 2: this tool goes to a different real website and gets a piece of advice
@tool("Get Life Advice")
def get_advice(topic: str = "any") -> str:
    """Fetches a random piece of life advice from a live advice API. Use this when the user wants advice. The topic parameter is optional."""
    response = requests.get("https://api.adviceslip.com/advice")
    data = response.json()
    advice_text = data['slip']['advice']
    return advice_text


print("Both tools are ready")

Both tools are ready


In [20]:
# Setting up the AI model that will power our agents
from crewai import Agent, Task, Crew, Process, LLM

my_llm = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.3)

print("The LLM is set up and ready to use")

The LLM is set up and ready to use


In [23]:
# TAsk :
# Creating two agents, each with their own role
from crewai import Agent

# Agent 1: this agent's job is to fetch content using our tools
fetcher_agent = Agent(
    role="Content Fetcher",
    goal="Fetch a joke and a piece of life advice using the available tools",
    backstory="You are a cheerful assistant who loves finding fun and useful content "
               "for people. You always use your tools to get real, fresh content "
               "instead of making things up.",
    tools=[get_random_joke, get_advice],
    llm=my_llm,
    allow_delegation=False,
    verbose=True
)

# Agent 2: this agent's job is to take the fetched content and write a nice message
writer_agent = Agent(
    role="Message Writer",
    goal="Take the joke and advice that were fetched, and turn them into a warm, friendly message",
    backstory="You are a skilled writer who takes raw pieces of content and turns them "
               "into something pleasant and easy to read for the end user.",
    tools=[],
    llm=my_llm,
    allow_delegation=True,
    verbose=True
)

print("Both agents are created")

Both agents are created


In [24]:
# Task 2 :
# Creating two tasks - one for each agent
from crewai import Task

# Task 1: the fetcher agent's job
fetch_task = Task(
    description="Fetch one joke and one piece of life advice using your tools.",
    expected_output="A joke and a piece of advice, clearly labeled.",
    agent=fetcher_agent
)

# Task 2: the writer agent's job - this uses Task 1's result as context
write_task = Task(
    description="Using the joke and advice that were fetched, write one short, "
                 "warm, friendly paragraph combining both into a nice message for the user.",
    expected_output="A short friendly paragraph combining the joke and the advice.",
    agent=writer_agent,
    context=[fetch_task]
)

print("Both tasks are created")

Both tasks are created


In [13]:
# Task 3 :
# a :
# Assemble the crew and run it - SEQUENTIAL process
from crewai import Crew, Process

my_crew = Crew(
    agents=[fetcher_agent, writer_agent],
    tasks=[fetch_task, write_task],
    process=Process.sequential,
    verbose=True
)

result = await my_crew.kickoff_async()

print("\n\nFINAL RESULT:")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 329045a2-4293-483b-a7c9-c0e7269811cd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Fetch one joke and one piece of life advice using your tools.                                            │
│  ID: cca808e1-8f8d-4eb1-a85b-3a4d2c894947                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│  Task: Fetch one joke and one piece of life advice using your tools.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_random_joke                                                                                          │
│  Args: {'topic': 'any'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_life_advice                                                                                          │
│  Args: {'topic': 'any'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Message Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As you go about your day, remember that it's okay to not take yourself too seriously and have a little fun -   │
│  after all, why did the orange lose the race? It ran out of juice! But in all seriousness, it's a good          │
│  reminder that no matter how old we get, we never really grow up, and that's what makes life so exciting -      │
│  there's always room to learn, explore, and have new experiences.                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the joke and advice that were fetched, write one short, warm, friendly paragraph combining both    │
│  into a nice message for the user.                                                                              │
│  Agent: Message Writer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



FINAL RESULT:
As you go about your day, remember that it's okay to not take yourself too seriously and have a little fun - after all, why did the orange lose the race? It ran out of juice! But in all seriousness, it's a good reminder that no matter how old we get, we never really grow up, and that's what makes life so exciting - there's always room to learn, explore, and have new experiences.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 329045a2-4293-483b-a7c9-c0e7269811cd                                                                       │
│  Final Output: As you go about your day, remember that it's okay to not take yourself too seriously and have a  │
│  little fun - after all, why did the orange lose the race? It ran out of juice! But in all seriousness, it's a  │
│  good reminder that no matter how old we get, we never really grow up, and that's what makes life so exciting   │
│  - there's always room to learn, explore, and have new experiences.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [25]:
# b :
# Run the crew again - HIERARCHICAL process (with a manager agent deciding order)
my_crew_hierarchical = Crew(
    agents=[fetcher_agent, writer_agent],
    tasks=[fetch_task, write_task],
    process=Process.hierarchical,
    manager_llm=my_llm,
    verbose=True
)

result_hierarchical = await my_crew_hierarchical.kickoff_async()

print("\n\nFINAL RESULT (hierarchical):")
print(result_hierarchical)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2158003e-f874-48f3-a587-fdc1a112e6c2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Fetch one joke and one piece of life advice using your tools.                                            │
│  ID: 29570959-6925-480d-8109-6c71c71966cb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Fetch one joke and one piece of life advice using your tools.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_random_joke                                                                                          │
│  Args: {'topic': 'any'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_life_advice                                                                                          │
│  Args: {'topic': 'any'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I just heard the most hilarious joke that I had to share with you - What kind of bagel can fly? The answer     │
│  is: A plain bagel. It's such a simple yet clever play on words, isn't it? On a more serious note, I wanted to  │
│  remind you of something important: when tackling challenges, remember not to cross the streams. It's a         │
│  valuable piece of advice that can help you navigate complex situations and avoid unnecessary complications. I  │
│  hope you have a wonderful day, filled with laughter and wisdom, and maybe even a delicious plain bagel or      │
│  two!                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the joke and advice that were fetched, write one short, warm, friendly paragraph combining both    │
│  into a nice message for the user.                                                                              │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



FINAL RESULT (hierarchical):
I just heard the most hilarious joke that I had to share with you - What kind of bagel can fly? The answer is: A plain bagel. It's such a simple yet clever play on words, isn't it? On a more serious note, I wanted to remind you of something important: when tackling challenges, remember not to cross the streams. It's a valuable piece of advice that can help you navigate complex situations and avoid unnecessary complications. I hope you have a wonderful day, filled with laughter and wisdom, and maybe even a delicious plain bagel or two!


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2158003e-f874-48f3-a587-fdc1a112e6c2                                                                       │
│  Final Output: I just heard the most hilarious joke that I had to share with you - What kind of bagel can fly?  │
│  The answer is: A plain bagel. It's such a simple yet clever play on words, isn't it? On a more serious note,   │
│  I wanted to remind you of something important: when tackling challenges, remember not to cross the streams.    │
│  It's a valuable piece of advice that can help you navigate complex situations and avoid unnecessary            │
│  complications. I hope you have a wonderful day, filled with laughter and wisdom, and maybe even a delicious    │
│  plain bagel or two!                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [28]:
# COMMON REQUIREMENT 4: Demonstrate on 3 distinct queries (query 2 of 3)
from crewai import Task, Crew, Process

query2_task = Task(
    description="Get me a joke to brighten my day.",
    expected_output="A single joke.",
    agent=fetcher_agent
)

query2_crew = Crew(agents=[fetcher_agent], tasks=[query2_task], process=Process.sequential, verbose=True)
query2_result = await query2_crew.kickoff_async()

print("\n\nQUERY 2 RESULT:")
print(query2_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 52b07e31-c8a8-4a05-9353-4a0ae3e0ce82                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Get me a joke to brighten my day.                                                                        │
│  ID: e0f65a1f-a294-49c8-a209-f2cb8972b639                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│  Task: Get me a joke to brighten my day.                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#6) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_random_joke                                                                                          │
│  Args: {'topic': 'any'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool get_random_joke executed with result: What do you get when you cross a React developer with a mathematician? — A function component....


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I hope that joke brightened your day!                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Get me a joke to brighten my day.                                                                        │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



QUERY 2 RESULT:
I hope that joke brightened your day!


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 52b07e31-c8a8-4a05-9353-4a0ae3e0ce82                                                                       │
│  Final Output: I hope that joke brightened your day!                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [29]:
# COMMON REQUIREMENT 4: Demonstrate on 3 distinct queries (query 3 of 3)
query3_task = Task(
    description="I'm feeling stuck in life. Can you give me some advice?",
    expected_output="A single piece of life advice.",
    agent=fetcher_agent
)

query3_crew = Crew(agents=[fetcher_agent], tasks=[query3_task], process=Process.sequential, verbose=True)
query3_result = await query3_crew.kickoff_async()

print("\n\nQUERY 3 RESULT:")
print(query3_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d34761d6-a10a-41e6-967a-08c049546c40                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: I'm feeling stuck in life. Can you give me some advice?                                                  │
│  ID: 38ce7c64-dfc1-46c5-88f8-a3488c861bfd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│  Task: I'm feeling stuck in life. Can you give me some advice?                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#30) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_life_advice                                                                                          │
│  Args: {'topic': 'any'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool get_life_advice executed with result: Work is never as important as you think it is....


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Work is never as important as you think it is.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: I'm feeling stuck in life. Can you give me some advice?                                                  │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



QUERY 3 RESULT:
Work is never as important as you think it is.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d34761d6-a10a-41e6-967a-08c049546c40                                                                       │
│  Final Output: Work is never as important as you think it is.                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [30]:
# PART 4, TASK 4: Confirm delegation (allow_delegation=True)
# The Message Writer agent has allow_delegation=True.
# During the Task 3a (sequential) run, the Writer agent used the
# delegate_work_to_coworker tool to ask the Content Fetcher for a
# combined draft, since the Writer itself has no tools.
# This is a real, captured example of delegation happening mid-task
# (see the "Tool Execution Started: delegate_work_to_coworker" log
# in the Task 3a output above).